# News Category Classification using BERT

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import torch
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re
import string
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import seaborn as sns
import matplotlib.pyplot as plt

# NLTK Downloads (required for text cleaning/lemmatization)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Mount Drive
drive.mount('/content/gdrive')

In [ ]:
# Load data
file_path = '/content/gdrive/MyDrive/News_Category_Dataset_v3.json'
df = pd.read_json(file_path, lines=True)

print(f"Total data points: {len(df)}")
print("\nFirst 5 rows:")
print(df.head())
print("\nCategory distribution (Top 10):")
print(df['category'].value_counts().head(10))

## 2. Preprocessing, Encoding, and Dataset Creation

In [ ]:
# Combine relevant text fields
df['combined_text'] = df['headline'] + " " + df['short_description']

# 1. Text Cleaning Functions
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('N'): return wordnet.NOUN
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def clean_and_lemmatize_text(text):
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?:\S*|www\.S*','',text)
    # Remove punctuation
    text = text.translate(str.maketrans('','', string.punctuation))
    # Remove stop words
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    text = " ".join(filtered_words)
    
    # Lemmatize
    tokens = word_tokenize(text)
    tagged_words = nltk.pos_tag(tokens)
    lemmatized_words = []
    for word, tag in tagged_words:
        w_pos = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, pos=w_pos)
        lemmatized_words.append(lemma)

    return " ".join(lemmatized_words)

# Apply cleaning and lemmatization
df['processed_text'] = df['combined_text'].apply(clean_and_lemmatize_text)
print("\nExample of processed text:")
print(df['processed_text'].iloc[0])

In [ ]:
# 2. Label Encoding
le = LabelEncoder()
df['category_id'] = le.fit_transform(df['category'])
num_classes = len(le.classes_)
category_map = {id: name for id, name in enumerate(le.classes_)}

print(f"Total Number of Classes: {num_classes}")
print(f"Example Category Mapping: {category_map[df['category_id'].iloc[0]]} -> {df['category_id'].iloc[0]}")

# 3. Split Data (using a small subset for quick fine-tuning demonstration)
X_train, X_val_test, y_train, y_val_test = train_test_split(
    df['processed_text'], df['category_id'], test_size=0.04, random_state=42, stratify=df['category_id']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test, test_size=0.5, random_state=42, stratify=y_val_test
)

# Create small subsets for demonstration (if the full dataset is too large)
small_train_df = pd.DataFrame({'text': X_train, 'label': y_train}).sample(n=15000, random_state=42)
small_val_df = pd.DataFrame({'text': X_val, 'label': y_val})

print(f"Train set size: {len(small_train_df)}")
print(f"Validation set size: {len(small_val_df)}")

In [ ]:
# 4. Tokenization and Hugging Face Dataset Creation

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_data(data_df):
    """Tokenizes a pandas DataFrame column and returns the encoded dataset."""
    encodings = tokenizer(
        data_df['text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=128
    )
    # Convert data into a format compatible with the Hugging Face Trainer
    class NewsDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item

        def __len__(self):
            return len(self.labels)

    return NewsDataset(encodings, data_df['label'].tolist())

small_train_dataset = tokenize_data(small_train_df)
val_dataset = tokenize_data(small_val_df)

print("Datasets prepared.")

## 3. BERT Fine-Tuning Setup and Training

In [ ]:
# 1. Define Compute Metrics Function (Essential for logging accuracy during training)
def compute_metrics(p):
    """Computes accuracy on the validation set."""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return {'accuracy': accuracy_score(labels, predictions)}

# 2. Initialize Model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", 
    num_labels=num_classes
).to(device)

# 3. Training Arguments (Configured for monitoring accuracy by epoch)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3, # Recommended 3 epochs
    per_device_train_batch_size=16, # Increased from 8 for stability/speed
    per_device_eval_batch_size=16,
    logging_dir='./logs',
    logging_steps=500,
    
    # Configuration to enable evaluation during training
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(), # Use mixed precision if GPU is available
)

# 4. Initialize DataCollator and Trainer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics, # Pass the metric function
)

print("Trainer initialized with epoch-based evaluation strategy.")

In [ ]:
# 5. Start Training
print("Starting training...")
trainer.train()

# 6. Final Evaluation
results = trainer.evaluate()
print("\nFinal Evaluation Results:")
print(results)

## 4. Detailed Analysis and Inference

In [ ]:
# 1. Detailed Classification Report
predictions_output = trainer.predict(val_dataset)
y_true = predictions_output.label_ids
y_pred = np.argmax(predictions_output.predictions, axis=1)

print("\n=== Detailed Classification Report ===")
print(classification_report(
    y_true, y_pred, 
    target_names=le.classes_,
    digits=4
))

# 2. Confusion Matrix for Visualization (Top 10 Classes)
plt.figure(figsize=(14, 12))
cm = confusion_matrix(y_true, y_pred)

# Get top 10 most frequent classes for a readable plot
top_classes_ids = small_val_df['label'].value_counts().nlargest(10).index.tolist()
top_classes_names = [category_map[i] for i in top_classes_ids]
top_cm = cm[top_classes_ids, :][:, top_classes_ids]

sns.heatmap(top_cm, annot=True, fmt='d',
            xticklabels=top_classes_names,
            yticklabels=top_classes_names,
            cmap='Blues')
plt.title('Confusion Matrix (Top 10 Most Frequent Classes)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# 3. Example Inference using a Pipeline
from transformers import pipeline

# Create the pipeline (device=0 uses GPU, -1 uses CPU)
classifier = pipeline(
    "text-classification", 
    model=model, 
    tokenizer=tokenizer, 
    device=0 if torch.cuda.is_available() else -1
)

# Test sentences
sample_news_1 = "Stocks surge as tech companies report record profits for the quarter."
sample_news_2 = "A new study links mindfulness to better sleep patterns in young adults."

results_1 = classifier(sample_news_1)[0]
predicted_id_1 = int(results_1['label'].split('_')[-1])
predicted_category_1 = category_map[predicted_id_1]

results_2 = classifier(sample_news_2)[0]
predicted_id_2 = int(results_2['label'].split('_')[-1])
predicted_category_2 = category_map[predicted_id_2]

print(f"\nSample 1: {sample_news_1}")
print(f"Predicted Category: {predicted_category_1} (Score: {results_1['score']:.4f})")
print(f"\nSample 2: {sample_news_2}")
print(f"Predicted Category: {predicted_category_2} (Score: {results_2['score']:.4f})")